# 01.8 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/08-statistics-by-hand/solutions.ipynb)

## Exercise 1

Sorted: 5, 7, 8, 12, 14, 15, 40. $n = 7$.
Mean $= 101/7 = 14.43$. Median = 4th value = 12.
Deviations²: 88.9, 55.2, 41.3, 5.9, 0.2, 0.3, 653.9 → sum 845.7; variance $= 845.7/6 = 140.95$; std $= 11.87$.
Lower half 5, 7, 8 → Q1 = 7; upper half 14, 15, 40 → Q3 = 15; IQR = 8; upper fence $= 15 + 12 = 27$. **40 is flagged.**

(Pandas interpolates and reports Q1 = 7.5, Q3 = 14.5; the fence moves to 25 and 40 is still flagged.)

In [1]:
import pandas as pd
s = pd.Series([5, 7, 8, 12, 14, 15, 40])
q1, q3 = s.quantile(.25), s.quantile(.75)
print(s.describe().round(2))
print("outliers:", list(s[(s < q1 - 1.5*(q3-q1)) | (s > q3 + 1.5*(q3-q1))]))

count     7.00
mean     14.43
std      11.87
min       5.00
25%       7.50
50%      12.00
75%      14.50
max      40.00
dtype: float64
outliers: [40]


In [2]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

(541910, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")

541,910 rows → 392,693 after cleaning


In [4]:
order_values = sales.groupby("Invoice")["Revenue"].sum()
customers = sales.groupby("Customer ID").agg(
    orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), items=("Quantity", "sum"))
customers["avg_order"] = customers["spend"] / customers["orders"]

## Exercise 2

In [5]:
country = sales.groupby("Customer ID")["Country"].first()
cs = customers.join(country)
top6 = sales.groupby("Country")["Revenue"].sum().nlargest(6).index
out = cs[cs["Country"].isin(top6)].groupby("Country")["spend"].agg(["count", "mean", "median"])
out["mean/median"] = out["mean"] / out["median"]
out.round(1).sort_values("mean/median", ascending=False)

,count,mean,median,mean/median
Country,,,,
Netherlands,9,31716.3,463.7,68.4
Australia,9,15538.2,1582.6,9.8
United Kingdom,3920,1858.4,645.0,2.9
France,87,2401.8,869.7,2.8
Germany,94,2432.7,1066.1,2.3
EIRE,3,88420.8,117210.1,0.8


The Netherlands has the biggest gap: nine customers, one of whom spent about £280k. The mean describes that one customer; the median describes the other eight.

## Exercise 3

In [6]:
def r_without_top(df, col, pct=0.01):
    cut = df["spend"].quantile(1 - pct)
    return df[col].corr(df["spend"]), df[df["spend"] < cut][col].corr(df["spend"])

for col in ["orders", "products"]:
    full, trimmed = r_without_top(customers, col)
    print(f"{col:<9} all: {full:.3f}   without top 1%: {trimmed:.3f}")

orders    all: 0.553   without top 1%: 0.754
products  all: 0.386   without top 1%: 0.570


## Exercise 4

In [7]:
def my_std(values):
    values = list(values)
    m = sum(values) / len(values)
    return (sum((v - m) ** 2 for v in values) / (len(values) - 1)) ** 0.5

sample = order_values.sample(1000, random_state=0)
print("mine:  ", round(my_std(sample), 4))
print("pandas:", round(sample.std(), 4))

mine:   2524.6273
pandas: 2524.6273
